# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll examine the schema and record sets, process records, and visualize key aspects of the data.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata  # Don't treat as dict or list, use as an object
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant metadata includes record sets, fields, and columns. We will list their `@id` attributes for reference.

In [ ]:
# Display available record sets and their fields using @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for fld in fields:
            if isinstance(fld, dict):
                print(f"    Field @id: {fld['@id']}, name: {fld.get('name', '')}")
            else:
                print(f"    Field @id: {fld}")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for col in columns:
            if isinstance(col, dict):
                print(f"    Column @id: {col['@id']}, name: {col.get('name', '')}")
            else:
                print(f"    Column @id: {col}")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets and fields are referenced by their `@id` attributes.

Below, we extract records for all available record sets. Adjust the list in `record_sets_ids` to include only those you wish to analyze.

In [ ]:
# Prepare a list of record set @ids
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]

# For demonstration, pick the first record set
if record_sets_ids:
    chosen_record_set_id = record_sets_ids[0]
else:
    chosen_record_set_id = None

dataframes = {}
for rsid in record_sets_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

# Show columns for the first record set
if chosen_record_set_id:
    print(f"Columns of record set {chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps like filtering records, normalizing numeric fields, and grouping data by attributes. All fields are referenced via their `@id`.

Select a numeric field for analysis, filter for high values, normalize, and group by another field.

In [ ]:
import numpy as np

# Example usage: manually define field @ids based on metadata overview
# Replace these with actual values printed in previous steps!

# Suppose chosen_record_set_id has these columns:
# ['cr:logLikelihood', 'cr:variable', 'cr:coefficient', 'cr:pValue', 'cr:standardError', ...]

numeric_field_id = 'cr:logLikelihood'  # Use the @id for the numeric field
group_field_id = 'cr:variable'         # Use the @id for grouping
threshold = -50                       # Set a meaningful threshold for logLikelihood (example)

df = dataframes.get(chosen_record_set_id, pd.DataFrame())
if numeric_field_id in df.columns:
    # Convert field to numeric if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouped aggregation
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot the distribution of log likelihood values and show mean log likelihood per variable.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(df[numeric_field_id].dropna(), bins=20, color='steelblue', edgecolor='k')
    plt.xlabel(numeric_field_id)
    plt.title('Distribution of Log Likelihood')
    plt.show()

    # Grouped bar plot
    if group_field_id in df.columns:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 5))
        plt.bar(grouped[group_field_id], grouped[numeric_field_id], color='orange')
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} per Variable")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook has guided an exploration of the FAIR² dataset using the Croissant schema and the `mlcroissant` library. We reviewed the available record sets and fields (referenced by their `@id`s), loaded and filtered the data, performed elementary EDA and normalization, and visualized distributions and key relationships.

Key findings:
- The dataset provides insights into adoption of knowledge management practices in rangeland communities, with potential for policy and intervention analysis.
- Missing values and structural biases should be considered in further analysis.
- All entities referenced via their Croissant `@id` ensure reproducibility and semantic clarity.

Continue exploration by adjusting field IDs or including additional record sets as needed.